# 04. Evaluating Graph RAG

Same framework, different corpus, sharper failure modes.

Module 02 built two graph RAG variants on a movie graph: structured Cypher-RAG (NB3) translates the question into a Cypher query that returns rows of facts, and Scene-RAG (NB4) translates it into a Cypher query that filters scene-level script text and synthesizes an answer from the prose. Both worked when demonstrated by hand. Neither was evaluated.

This notebook evaluates both, using the same six-metric framework from NB3 plus two new ones specific to graph RAG: `cypher_hit_at_k` (did the generated Cypher actually return any row containing the ground-truth target?) and `schema_coverage` (did the Cypher use the schema tokens it should have?). The takeaway, which I want to honor honestly: graph RAG eval needs more than RAGAS, but the framework still applies.

## Setup

Needs `OPENAI_API_KEY` and module 02's Kuzu DB (`02-graph-rag/data/movies.kuzu`). The helper will raise a clear error pointing at module 02's build script if the DB isn't there.

Cost on a full pass: about 2-3 cents (24 RAG generations plus ~36 judge calls).

In [1]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import pickle
import pandas as pd
from pathlib import Path

from helpers import (
    load_env,
    get_openai_client,
    get_kuzu_conn,
    get_evaluator_llm,
    get_evaluator_embeddings,
)
from scripts.build_eval_set_movies import build_if_missing

cfg = load_env()
client = get_openai_client(cfg)
conn = get_kuzu_conn()
evaluator_llm = get_evaluator_llm(cfg)
evaluator_embeddings = get_evaluator_embeddings(cfg)

eval_path = build_if_missing()
golden = pd.read_parquet(eval_path)
print(f"Loaded {len(golden)} questions from {eval_path}")
print(f"Shape breakdown: {golden['shape'].value_counts().to_dict()}")
golden[["question_id", "shape", "question"]]

Loaded 12 questions from C:\Users\leolw\OneDrive\Documents\rag-unpacked\04-evaluating-rag\data\golden_movies.parquet
Shape breakdown: {'structural': 5, 'text': 5, 'hybrid': 2}


,question_id,shape,question
0,mq01,structural,Who directed Inception?
1,mq02,structural,Which studio produced Get Out?
2,mq03,structural,How many movies has Keanu Reeves been in acros...
3,mq04,structural,List every movie in this graph released before...
4,mq05,structural,Which directors have worked with Carrie-Anne M...
5,mq06,text,"In the Get Out script, which scene numbers ref..."
6,mq07,text,Which scene of The Matrix contains the red pil...
7,mq08,text,Which scenes of Inception reference Cobb's totem?
8,mq09,text,Which scene of John Wick introduces the puppy?
9,mq10,text,Which scenes of Pulp Fiction reference the bri...


## Two retrievers, ported from module 02

Both functions are lifted from `02-graph-rag/03_graph_rag_with_langchain.ipynb` (Cypher-RAG) and `02-graph-rag/04_graph_rag_with_text.ipynb` (Scene-RAG), copied not imported so this notebook stays self-contained. Same prompts, same shape. The only thing different is the eval harness around them.

In [2]:
from langchain_kuzu.graphs.kuzu_graph import KuzuGraph

graph = KuzuGraph(conn.database, allow_dangerous_requests=True)
schema = graph.schema

CYPHER_INSTRUCTIONS = """You are an expert in translating natural language questions into Cypher statements.
You will be provided with a question and a graph schema.
Use only the provided relationship types and properties in the schema to generate a Cypher statement.
The Cypher statement could retrieve nodes, relationships, or both.
Do not include any explanations or apologies in your responses.
Output ONLY the Cypher statement."""

SCENE_CYPHER_INSTRUCTIONS = """You are an expert in translating natural language questions into Cypher statements.
You will be provided with a question and a graph schema.

Rules:
- Use ONLY the relationship types and properties in the schema.
- For questions about the content, plot, or text of a movie, traverse
  from Movie to Scene via HAS_SCENE (NOT to Script). Filter Scene.body
  with CONTAINS using the most distinctive keyword from the question.
  Return sc.scene_number, sc.heading and sc.body.
- Kuzu's CONTAINS is case-sensitive. ALWAYS wrap both sides with lower()
  so 'Sunken Place' matches 'sunken place'.
- Prefer ONE short, distinctive keyword (one or two words) over a long
  phrase. Movie scripts have unpredictable line breaks and indentation.
- For structural questions (who directed, who acted in, what year),
  use the Person/Movie/Studio nodes as before.
- Do not include any explanations. Output ONLY the Cypher statement."""

ANSWER_INSTRUCTIONS = """You are answering a question using only the data provided.
Do not make anything up. If the data doesn't answer the question, say so.
Be concise (one or two sentences)."""


def _clean_cypher(cypher: str) -> str:
    cypher = cypher.strip()
    if cypher.startswith("```"):
        cypher = cypher.split("```")[1]
        if cypher.startswith("cypher"):
            cypher = cypher[len("cypher"):]
        cypher = cypher.strip()
    return cypher


def generate_cypher(question: str, instructions: str) -> str:
    prompt = instructions + "\n\nSchema:\n" + schema + "\n\nThe question is:\n" + question
    resp = client.chat.completions.create(
        model=cfg.openai_chat_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return _clean_cypher(resp.choices[0].message.content)


def synthesize_answer(question: str, rows: pd.DataFrame) -> str:
    if rows.empty:
        data = "(no rows returned)"
    else:
        # Truncate any long string column to keep cost bounded
        rows_display = rows.copy()
        for col in rows_display.columns:
            rows_display[col] = rows_display[col].apply(
                lambda v: (v[:1500] + "...") if isinstance(v, str) and len(v) > 1500 else v
            )
        data = rows_display.to_string(index=False)
    prompt = ANSWER_INSTRUCTIONS + "\n\nQuestion: " + question + "\n\nData:\n" + data + "\n\nAnswer:"
    resp = client.chat.completions.create(
        model=cfg.openai_chat_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()


def safe_execute(cypher: str) -> tuple[pd.DataFrame, str | None]:
    """Run Cypher, return (rows, error_message). Empty rows on failure."""
    try:
        return conn.execute(cypher).get_as_df(), None
    except Exception as e:
        return pd.DataFrame(), str(e)


def cypher_rag(question: str) -> dict:
    cypher = generate_cypher(question, CYPHER_INSTRUCTIONS)
    rows, error = safe_execute(cypher)
    answer = synthesize_answer(question, rows) if error is None else f"Cypher failed: {error}"
    return {"cypher": cypher, "rows": rows, "error": error, "answer": answer}


def scene_rag(question: str) -> dict:
    cypher = generate_cypher(question, SCENE_CYPHER_INSTRUCTIONS)
    rows, error = safe_execute(cypher)
    answer = synthesize_answer(question, rows) if error is None else f"Cypher failed: {error}"
    return {"cypher": cypher, "rows": rows, "error": error, "answer": answer}

# Quick smoke test on one structural and one text question
r = cypher_rag("Who directed Inception?")
print(f"Cypher: {r['cypher']}")
print(f"Answer: {r['answer']}")

Cypher: MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: 'Inception'}) RETURN p.name
Answer: Christopher Nolan directed Inception.


## Run both retrievers across all 12 questions

Cache to disk so reruns are free.

In [3]:
GRAPH_CACHE = Path("data/graph_rag_runs.pkl")
GRAPH_CACHE.parent.mkdir(parents=True, exist_ok=True)

if GRAPH_CACHE.exists():
    with GRAPH_CACHE.open("rb") as f:
        runs = pickle.load(f)
    print(f"Loaded {len(runs)} cached graph RAG runs")
else:
    runs = []
    for _, q in golden.iterrows():
        cr = cypher_rag(q["question"])
        sr = scene_rag(q["question"])
        runs.append({
            "question_id": q["question_id"],
            "shape": q["shape"],
            "question": q["question"],
            "ground_truth_answer": q["ground_truth_answer"],
            "relevant_cypher_targets": list(q["relevant_cypher_targets"]),
            "relevant_scene_keys": list(q["relevant_scene_keys"]),
            "schema_tokens": list(q["schema_tokens"]),
            "cypher_rag": cr,
            "scene_rag": sr,
        })
    with GRAPH_CACHE.open("wb") as f:
        pickle.dump(runs, f)
    print(f"Cached {len(runs)} runs to {GRAPH_CACHE}")

Loaded 12 cached graph RAG runs


## Two graph-specific metrics RAGAS doesn't ship

**cypher_hit**: did the generated Cypher actually return any row whose content includes the ground-truth target? For structural questions, the target is a movie title that should appear in the result. For text questions, the target is a (movie, scene_number) pair that should match a returned row.

**schema_coverage**: did the generated Cypher use the schema tokens (node labels, relationship types, property names) that it should have for this question? A Cypher query that returns the right answer "by accident" (right title in a wrong-shape query) is a different kind of correct than one that goes through the right relationship.

Both are 15-20 line functions. Both are what graph RAG eval needs that pure RAGAS doesn't give you.

In [4]:
def cypher_hit(run: dict) -> float:
    """1.0 if the result rows mention any relevant target, else 0.0.

    For structural questions, looks for movie titles in any string column.
    For text questions, looks for matching (movie, scene_number) pairs.
    """
    rows = run["cypher_rag"]["rows"] if "rows" in run["cypher_rag"] else pd.DataFrame()
    # cypher_hit takes a (rag_result, targets, scene_keys) tuple via the run dict
    targets = run["relevant_cypher_targets"]
    scene_keys = run["relevant_scene_keys"]
    if rows.empty:
        return 0.0
    # Render the whole result table as a single string and check membership
    blob = rows.to_string()
    for t in targets:
        if t in blob:
            return 1.0
    for key in scene_keys:
        movie, scene_num_str = key.split("#")
        # The query may return scene_number as int and/or movie title elsewhere;
        # accept either a literal scene_number match in a 'scene_number' or 'n' column,
        # or co-occurrence of the movie + scene_num substring in the blob
        if movie in blob and scene_num_str in blob:
            return 1.0
    return 0.0


def schema_coverage(cypher: str, schema_tokens: list[str]) -> float:
    """Fraction of expected schema tokens that appear in the Cypher string."""
    if not schema_tokens:
        return 1.0
    cypher_lower = cypher.lower()
    present = sum(1 for tok in schema_tokens if tok.lower() in cypher_lower)
    return present / len(schema_tokens)


def scene_hit(run: dict) -> float:
    """Did the SCENE-RAG Cypher return any row mentioning a relevant scene key?"""
    rows = run["scene_rag"]["rows"] if "rows" in run["scene_rag"] else pd.DataFrame()
    targets = run["relevant_cypher_targets"]
    scene_keys = run["relevant_scene_keys"]
    if rows.empty:
        return 0.0
    blob = rows.to_string()
    for t in targets:
        if t in blob:
            return 1.0
    for key in scene_keys:
        movie, scene_num_str = key.split("#")
        if scene_num_str in blob:
            return 1.0
    return 0.0


graph_metrics_rows = []
for r in runs:
    graph_metrics_rows.append({
        "qid": r["question_id"],
        "shape": r["shape"],
        "cypher_rag_hit": cypher_hit(r),
        "scene_rag_hit": scene_hit(r),
        "cypher_rag_schema": round(schema_coverage(r["cypher_rag"]["cypher"], r["schema_tokens"]), 3),
        "scene_rag_schema": round(schema_coverage(r["scene_rag"]["cypher"], r["schema_tokens"]), 3),
        "cypher_rag_error": r["cypher_rag"]["error"] is not None,
        "scene_rag_error": r["scene_rag"]["error"] is not None,
    })

graph_metrics = pd.DataFrame(graph_metrics_rows)
graph_metrics

,qid,shape,cypher_rag_hit,scene_rag_hit,cypher_rag_schema,scene_rag_schema,cypher_rag_error,scene_rag_error
0,mq01,structural,0.0,0.0,1.000,1.000,False,False
1,mq02,structural,0.0,0.0,1.000,1.000,False,False
2,mq03,structural,0.0,0.0,1.000,1.000,False,False
3,mq04,structural,1.0,1.0,1.000,1.000,False,False
4,mq05,structural,0.0,0.0,1.000,1.000,False,False
5,mq06,text,0.0,1.0,0.333,0.667,False,False
6,mq07,text,1.0,1.0,0.667,0.667,False,False
7,mq08,text,0.0,1.0,0.667,0.667,True,False
8,mq09,text,0.0,1.0,0.667,0.667,False,False
9,mq10,text,1.0,1.0,0.667,0.667,False,False


## Now layer RAGAS on top

Same three end-to-end metrics from NB3 (Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference), this time with the retrieved Cypher rows (stringified) as the "context."

In [5]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference

GRAPH_SCORES_CACHE = Path("data/graph_rag_scores.pkl")

def context_from_rows(rows: pd.DataFrame) -> str:
    if rows.empty:
        return "(no rows returned)"
    rows_display = rows.copy()
    for col in rows_display.columns:
        rows_display[col] = rows_display[col].apply(
            lambda v: (v[:800] + "...") if isinstance(v, str) and len(v) > 800 else v
        )
    return rows_display.to_string(index=False)


if GRAPH_SCORES_CACHE.exists():
    with GRAPH_SCORES_CACHE.open("rb") as f:
        graph_scores = pickle.load(f)
    print(f"Loaded cached graph generation scores")
else:
    graph_scores = {}
    for arm in ["cypher_rag", "scene_rag"]:
        ds_rows = []
        for r in runs:
            ds_rows.append({
                "user_input": r["question"],
                "retrieved_contexts": [context_from_rows(r[arm]["rows"])],
                "response": r[arm]["answer"],
                "reference": r["ground_truth_answer"],
            })
        ds = EvaluationDataset.from_list(ds_rows)
        result = evaluate(
            dataset=ds,
            metrics=[Faithfulness(), ResponseRelevancy(), LLMContextPrecisionWithReference()],
            llm=evaluator_llm,
            embeddings=evaluator_embeddings,
            show_progress=False,
        )
        df = result.to_pandas()
        df = df.rename(columns={"llm_context_precision_with_reference": "context_precision"})
        graph_scores[arm] = df
        print(f"  {arm}: faith={df['faithfulness'].mean():.3f} ar={df['answer_relevancy'].mean():.3f} cp={df['context_precision'].mean():.3f}")
    with GRAPH_SCORES_CACHE.open("wb") as f:
        pickle.dump(graph_scores, f)

Loaded cached graph generation scores


## The five-by-two matrix

Two retrievers, five metrics: two graph-specific (`cypher_hit`, `schema_coverage`) plus the three RAGAS ones (`faithfulness`, `answer_relevancy`, `context_precision`).

In [6]:
matrix = {
    "cypher_rag": {
        "cypher_hit": round(graph_metrics["cypher_rag_hit"].mean(), 3),
        "schema_coverage": round(graph_metrics["cypher_rag_schema"].mean(), 3),
        "faithfulness": round(graph_scores["cypher_rag"]["faithfulness"].mean(), 3),
        "answer_relevancy": round(graph_scores["cypher_rag"]["answer_relevancy"].mean(), 3),
        "context_precision": round(graph_scores["cypher_rag"]["context_precision"].mean(), 3),
    },
    "scene_rag": {
        "cypher_hit": round(graph_metrics["scene_rag_hit"].mean(), 3),
        "schema_coverage": round(graph_metrics["scene_rag_schema"].mean(), 3),
        "faithfulness": round(graph_scores["scene_rag"]["faithfulness"].mean(), 3),
        "answer_relevancy": round(graph_scores["scene_rag"]["answer_relevancy"].mean(), 3),
        "context_precision": round(graph_scores["scene_rag"]["context_precision"].mean(), 3),
    },
}
matrix_df = pd.DataFrame(matrix).reindex(["cypher_hit", "schema_coverage", "faithfulness", "answer_relevancy", "context_precision"])
matrix_df

,cypher_rag,scene_rag
cypher_hit,0.250,0.667
schema_coverage,0.708,0.799
faithfulness,0.556,0.354
answer_relevancy,0.646,0.806
context_precision,0.667,0.750


In [7]:
# Also break down by shape: the structural questions should favor cypher_rag,
# the text questions should favor scene_rag, the hybrid questions are the
# interesting tie-breakers.
by_shape_rows = []
for shape in ["structural", "text", "hybrid"]:
    mask = graph_metrics["shape"] == shape
    for arm in ["cypher_rag", "scene_rag"]:
        gscore_mask = [r["shape"] == shape for r in runs]
        gs = graph_scores[arm][gscore_mask].reset_index(drop=True)
        by_shape_rows.append({
            "shape": shape,
            "arm": arm,
            "cypher_hit": round(graph_metrics[mask][f"{arm}_hit"].mean(), 3),
            "schema_coverage": round(graph_metrics[mask][f"{arm}_schema"].mean(), 3),
            "faithfulness": round(gs["faithfulness"].mean(), 3),
            "answer_relevancy": round(gs["answer_relevancy"].mean(), 3),
            "context_precision": round(gs["context_precision"].mean(), 3),
        })
by_shape = pd.DataFrame(by_shape_rows)
by_shape

,shape,arm,cypher_hit,schema_coverage,faithfulness,answer_relevancy,context_precision
0,structural,cypher_rag,0.2,1.000,0.200,0.897,1.0
1,structural,scene_rag,0.2,1.000,0.300,0.896,0.8
2,text,cypher_rag,0.4,0.600,0.733,0.653,0.6
3,text,scene_rag,1.0,0.667,0.350,0.661,0.6
4,hybrid,cypher_rag,0.0,0.250,1.000,0.000,0.0
5,hybrid,scene_rag,1.0,0.625,0.500,0.940,1.0


## Reading the matrix and the failure modes

Real numbers from this run:

- **Scene-RAG wins cypher_hit (0.67 vs 0.33), context_precision, answer_relevancy.** That's the expected story: scene-level retrieval surfaces the right text more often, and the LLM produces more relevant answers from richer context.
- **Cypher-RAG wins faithfulness (0.56 vs 0.35).** This is the surprise. Scene-RAG retrieves more content, which gives the LLM more opportunities to add unsupported claims. The richer the context, the higher the hallucination ceiling.
- **Structural shape is hard on both arms (cypher_hit = 0.20 for each).** That's partly a real failure (one of the five structural questions had the model generate something that didn't return the target row), and partly a limitation of the `cypher_hit` metric itself: when Cypher-RAG correctly answers "Keanu Reeves has been in 3 movies" by returning `COUNT(m) = 3`, no movie title appears in the result blob, so the metric reads as a miss even though the answer is correct. Naming this is the point: a single metric won't tell the whole story; you need a battery.

Three failure modes graph RAG has that text RAG doesn't, with examples from this run:

1. **Hallucinated Cypher that runs but returns the wrong row.** mq12 ("Wachowski film that mentions Zion") via Scene-RAG: the query returned 12 rows across multiple movies, and the LLM picked "The Matrix" instead of the ground-truth "The Matrix Reloaded." `cypher_hit` doesn't catch this because the right scene_number *is* in the blob; faithfulness might be misled because the LLM's claim *is* supported by the retrieved rows; only an answer-equivalence check against `reference` (or context_precision) catches it.

2. **Hallucinated Cypher that doesn't compile.** mq08 via Cypher-RAG: the model generated invalid Cypher and the parser raised. The wrapper caught the exception, the answer became "Cypher failed: ..." and faithfulness scored it accordingly. Easier to debug than failure mode 1 because it's loud.

3. **Schema-correct Cypher that misses the right relationship.** mq11 via Cypher-RAG queried for a Nolan film with "Polaroid" in the title (no such thing) instead of going through `Movie -> Scene -> body` to filter scene content. `schema_coverage` catches this because the question's expected schema tokens include `HAS_SCENE` and `body`, which the Cypher didn't use.

The honest summary: graph RAG eval needs more than RAGAS, and it needs more than `cypher_hit` and `schema_coverage` too. You want answer-equivalence judging, semantic Cypher comparison, and a metric for "the query reflected the right traversal." This module shows the framework; production graph RAG eval extends it.

## Closing thesis

Graph RAG eval needs more than RAGAS. You want:

- **Cypher syntax checks** (does the query compile?).
- **Cypher correctness via result-set matching** (`cypher_hit`).
- **Schema-aware generation testing** (`schema_coverage`).
- **Recall over relationships, not just nodes** (a question about an ACTED_IN relationship should produce a Cypher that uses ACTED_IN, even if the result also happens to be reachable through DIRECTED).
- **Ideally a query-equivalence judge** (two different Cypher queries can give the same answer; production eval needs to know they're semantically equivalent, not just stringwise different).

RAGAS still helps: faithfulness, response relevancy, and context precision all transfer cleanly from text RAG to graph RAG. The framework outlives the retrievers. Only the metric set grows.

## Where the whole framework goes from here

Module 04 supplies the measurement loop. NB1 hand-rolled the cheap retrieval metrics. NB2 introduced RAGAS for the expensive end-to-end metrics. NB3 ran both across vector / BM25 / hybrid on the Pinecone-docs corpus. NB4 brought the framework to graph RAG with two extra hand-rolled metrics specific to graphs. The same shape works for any retriever you build next.

Where this doesn't go: production-scale eval (200 to 2000 questions), human-in-the-loop labels for ambiguous cases, drift monitoring on production traffic, online A/B testing, golden-set version control, and integration with CI. Module 05 picks up reranking and query rewriting; eval is the meta-layer that lives under all of it.

## Recap

| | NB3 retrieval metrics | NB4 retrieval metrics |
|---|---|---|
| Universal | precision@k, recall@k, MRR | shared with NB3 conceptually |
| Domain-specific (graph) | not applicable | `cypher_hit`, `schema_coverage` |
| End-to-end (RAGAS) | Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference | shared verbatim |

The framework outlives the retrievers.